# CNN Feature Extractor + Classical ML Ensemble

**Pipeline:**
1. Build a custom CNN (128×128 input) with a Dense(256) feature-extraction layer
2. Train the CNN end-to-end (10 epochs)
3. Use the `dense` layer output (256 features) to train RF / DT / SVM
4. Combine classifiers into a soft-vote ensemble
5. Evaluate and save `cnn_ensemble.h5` + `cnn_ensemble_model.pkl`

> Input size 128×128 keeps the CNN backbone consistent with the Xception+Ensemble
> and InceptionV3+Ensemble notebooks, and avoids out-of-memory errors.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
import glob
import joblib

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import label_binarize

print('TensorFlow:', tf.__version__)
print('GPUs available:', tf.config.list_physical_devices('GPU'))

## 1. Load Dataset (128×128, RGB, float32)

In [ ]:
IMAGE_SIZE = 128

train_images = []
train_labels = []

for directory_path in glob.glob(os.path.join('..', 'MRI_DATASET', 'Training', '*')):
    label = os.path.basename(directory_path)
    print('Loading class:', label)
    for img_path in glob.glob(os.path.join(directory_path, '*')):
        if img_path.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
            img = cv2.imread(img_path, cv2.IMREAD_COLOR)
            img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            train_images.append(img)
            train_labels.append(label)

# Single normalisation — divide once, keep float32 to halve memory vs float64
train_images = np.array(train_images, dtype=np.float32) / 255.0
train_labels = np.array(train_labels)

label_to_id = {v: i for i, v in enumerate(np.unique(train_labels))}
id_to_label = {v: k for k, v in label_to_id.items()}
train_label_ids = np.array([label_to_id[x] for x in train_labels])

print(f'Training: {len(train_images)} images')
print('Label mapping:', label_to_id)

In [ ]:
test_images = []
test_labels = []

for directory_path in glob.glob(os.path.join('..', 'MRI_DATASET', 'Testing', '*')):
    label = os.path.basename(directory_path)
    for img_path in glob.glob(os.path.join(directory_path, '*')):
        if img_path.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
            img = cv2.imread(img_path, cv2.IMREAD_COLOR)
            img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            test_images.append(img)
            test_labels.append(label)

test_images  = np.array(test_images,  dtype=np.float32) / 255.0
test_labels  = np.array(test_labels)
test_label_ids = np.array([label_to_id[x] for x in test_labels])

print(f'Testing: {len(test_images)} images')

## 2. Build CNN with Dense(256) Feature-Extraction Layer

Architecture mirrors the standalone CNN (`CNN/cnn_model.ipynb`) but uses:
- **128×128** input (saves memory, keeps feature extractor compact)
- **Dense(256, relu)** as the named feature layer (`'dense'`) instead of Dense(64)
- Dropout(0.5) for regularisation

The layer name `dense` is what `model_loader.py` references as `FEATURE_LAYERS['cnn_feat']`.

In [ ]:
N_CLASSES = 4
input_shape = (IMAGE_SIZE, IMAGE_SIZE, 3)

cnn_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    # Feature-extraction layer — name 'dense' is referenced by model_loader.py
    layers.Dense(256, activation='relu', name='dense'),
    layers.Dropout(0.5),
    layers.Dense(N_CLASSES, activation='softmax', name='dense_1'),
], name='cnn_feature_extractor')

cnn_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

cnn_model.summary()

## 3. Train the CNN (10 epochs)

In [ ]:
history = cnn_model.fit(
    train_images, train_label_ids,
    batch_size=32,
    epochs=10,
    verbose=1,
    validation_data=(test_images, test_label_ids)
)

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate CNN standalone accuracy on the official test set
cnn_loss, cnn_acc = cnn_model.evaluate(test_images, test_label_ids, verbose=0)
print(f'CNN standalone test accuracy: {cnn_acc * 100:.2f}%')

## 4. Save the CNN Backbone (`cnn_ensemble.h5`)

This file is loaded by `model_loader.py` as the feature extractor for the ensemble.

In [ ]:
cnn_model.save('../MODELS/cnn_ensemble.h5')
print('Saved: ../MODELS/cnn_ensemble.h5')

## 5. Extract Dense(256) Features

Create a sub-model that outputs from the `dense` layer, then batch-predict features
for both training and test sets.

In [ ]:
# Sub-model: input → Dense(256) activations
feature_extractor = Model(
    inputs=cnn_model.input,
    outputs=cnn_model.get_layer('dense').output
)

# Batch-predict to avoid a second OOM risk
train_features = feature_extractor.predict(train_images, batch_size=64, verbose=1)
test_features  = feature_extractor.predict(test_images,  batch_size=64, verbose=1)

print('Train features shape:', train_features.shape)  # (6984, 256)
print('Test  features shape:', test_features.shape)   # (2063, 256)

## 6. Train Classical Classifiers on CNN Features

In [ ]:
# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(train_features, train_label_ids)

pred_rf = rf.predict(test_features)
acc_rf  = accuracy_score(test_label_ids, pred_rf)
print(f'Random Forest accuracy: {acc_rf * 100:.2f}%')

In [ ]:
print('Random Forest — Classification Report:')
print(classification_report(test_label_ids, pred_rf,
                            target_names=[id_to_label[i] for i in range(N_CLASSES)]))

In [ ]:
# --- Decision Tree ---
dt = DecisionTreeClassifier(random_state=42)
dt.fit(train_features, train_label_ids)

pred_dt = dt.predict(test_features)
acc_dt  = accuracy_score(test_label_ids, pred_dt)
print(f'Decision Tree accuracy: {acc_dt * 100:.2f}%')

In [ ]:
print('Decision Tree — Classification Report:')
print(classification_report(test_label_ids, pred_dt,
                            target_names=[id_to_label[i] for i in range(N_CLASSES)]))

In [ ]:
# --- SVM (with probability=True so soft voting works) ---
svm = SVC(probability=True, random_state=42)
svm.fit(train_features, train_label_ids)

pred_svm = svm.predict(test_features)
acc_svm  = accuracy_score(test_label_ids, pred_svm)
print(f'SVM accuracy: {acc_svm * 100:.2f}%')

In [ ]:
print('SVM — Classification Report:')
print(classification_report(test_label_ids, pred_svm,
                            target_names=[id_to_label[i] for i in range(N_CLASSES)]))

## 7. Soft-Vote Ensemble (RF + DT + SVM)

In [ ]:
ensemble_model = VotingClassifier(
    estimators=[('rf', rf), ('dt', dt), ('svm', svm)],
    voting='soft'
)
ensemble_model.fit(train_features, train_label_ids)

pred_ens = ensemble_model.predict(test_features)
acc_ens  = accuracy_score(test_label_ids, pred_ens)
print(f'Ensemble (RF+DT+SVM) accuracy: {acc_ens * 100:.2f}%')

In [ ]:
print('Ensemble — Classification Report:')
print(classification_report(test_label_ids, pred_ens,
                            target_names=[id_to_label[i] for i in range(N_CLASSES)]))

## 8. Confusion Matrices

In [ ]:
class_names = [id_to_label[i] for i in range(N_CLASSES)]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Confusion Matrices — CNN Features', fontsize=14)

for ax, preds, title in zip(
    axes.ravel(),
    [pred_rf, pred_dt, pred_svm, pred_ens],
    ['Random Forest', 'Decision Tree', 'SVM', 'Soft-Vote Ensemble']
):
    cm = confusion_matrix(test_label_ids, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 9. Accuracy Summary

In [ ]:
model_names = ['Random Forest', 'Decision Tree', 'SVM', 'Ensemble (RF+DT+SVM)']
accuracies  = [acc_rf, acc_dt, acc_svm, acc_ens]

plt.figure(figsize=(10, 5))
bars = plt.bar(model_names, [a * 100 for a in accuracies],
               color=['steelblue', 'salmon', 'seagreen', 'mediumpurple'])
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.5,
             f'{acc * 100:.2f}%', ha='center', va='bottom', fontweight='bold')
plt.ylim(0, 110)
plt.ylabel('Test Accuracy (%)')
plt.title('CNN Feature Extractor + Classical ML — Accuracy Comparison')
plt.tight_layout()
plt.show()

print('\nSummary:')
for name, acc in zip(model_names, accuracies):
    print(f'  {name:30s}: {acc * 100:.2f}%')

## 10. Save the Classical Ensemble (`cnn_ensemble_model.pkl`)

In [ ]:
joblib.dump(ensemble_model, '../MODELS/cnn_ensemble_model.pkl')
print('Saved: ../MODELS/cnn_ensemble_model.pkl')